# Notebook 06: Basic Quantum Gates

Quantum gates are the operations that transform quantum states.  Every gate is
a **unitary matrix** -- it preserves the norm of the state vector (probabilities
still sum to 1 after the gate acts).

This notebook covers:
1. Single-qubit gates: H (Hadamard), X (NOT), Z (phase flip)
2. Verifying unitarity and norm preservation
3. Phase changes: same probabilities, different amplitudes
4. The CNOT gate on 2-qubit states
5. `expand_single_qubit_gate` for multi-qubit operations
6. **Demo C:** The oracle phase flip -- invisible to probabilities, but amplified by diffusion

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
from quantum_demo.linalg import ket, is_normalized, is_unitary, dagger
from quantum_demo.states import qubit_state, amplitudes_to_probabilities, pretty_basis_labels, equal_superposition
from quantum_demo.gates import H, X, Y, Z, S, T, CNOT, I2, apply_gate, phase_oracle, diffusion_operator
from quantum_demo.tensor import tensor, basis_state_bits, expand_single_qubit_gate
from quantum_demo.viz.static_plots import plot_probabilities, plot_real_amplitudes
from quantum_demo.viz.state_plots import plot_oracle_phase_demo

## 1. Meet the gates

Let's print the matrix form of each single-qubit gate.

In [ ]:
gates = {'I': I2, 'X': X, 'Y': Y, 'Z': Z, 'H': H, 'S': S, 'T': T}

for name, gate in gates.items():
    print(f"--- {name} gate ---")
    print(np.array2string(gate, precision=4, suppress_small=True))
    print()

### What does each gate do?

| Gate | Matrix | Action |
|------|--------|--------|
| **X** | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | Bit flip: $|0\rangle \leftrightarrow |1\rangle$ |
| **Z** | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | Phase flip: $|1\rangle \to -|1\rangle$ |
| **H** | $\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | Creates superposition from basis states |

## 2. Verifying unitarity

A matrix $U$ is unitary if $U^\dagger U = I$.  This is the quantum analogue
of saying "the operation is reversible and preserves probabilities."

In [ ]:
for name, gate in gates.items():
    print(f"{name}: is_unitary = {is_unitary(gate)}")

In [ ]:
# Let's verify explicitly for H: H^dagger @ H should equal I
product = dagger(H) @ H
print("H^dagger @ H =")
print(np.array2string(product, precision=4, suppress_small=True))
print("\nClose to identity?", np.allclose(product, np.eye(2)))

## 3. Gate action on single-qubit states

Let's see how H, X, and Z transform the basis states.

In [ ]:
ket0 = ket(0, 2)
ket1 = ket(1, 2)

print("=== X gate (bit flip) ===")
print(f"X|0> = {apply_gate(ket0, X)}  (should be |1>)")
print(f"X|1> = {apply_gate(ket1, X)}  (should be |0>)")

print("\n=== H gate (superposition) ===")
h_ket0 = apply_gate(ket0, H)
h_ket1 = apply_gate(ket1, H)
print(f"H|0> = {h_ket0}  (= |+>)")
print(f"H|1> = {h_ket1}  (= |->)")

print("\n=== Z gate (phase flip) ===")
print(f"Z|0> = {apply_gate(ket0, Z)}  (unchanged)")
print(f"Z|1> = {apply_gate(ket1, Z)}  (picks up minus sign)")

## 4. Norm preservation

Because gates are unitary, applying them never changes whether a state is
normalized.  Let's verify this for several transformations.

In [ ]:
# Start with an arbitrary normalized qubit state
psi = qubit_state(3, 4)  # normalized version of [3, 4]
print(f"Initial state: {psi}")
print(f"Normalized? {is_normalized(psi)}")

# Apply a sequence of gates
psi_after_H = apply_gate(psi, H)
psi_after_HZ = apply_gate(psi_after_H, Z)
psi_after_HZX = apply_gate(psi_after_HZ, X)

print(f"\nAfter H:   {psi_after_H}  normalized? {is_normalized(psi_after_H)}")
print(f"After HZ:  {psi_after_HZ}  normalized? {is_normalized(psi_after_HZ)}")
print(f"After HZX: {psi_after_HZX}  normalized? {is_normalized(psi_after_HZX)}")

## 5. Phase changes: same probabilities, different states

This is one of the most important insights in quantum mechanics.

Consider the state $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ and apply
the Z gate.  Z flips the sign of the $|1\rangle$ component:

$$Z|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle) = |-\rangle$$

The *probabilities* of measuring 0 or 1 are identical (both 50%).  But the
*amplitudes* are different.  This hidden phase information is the resource
that quantum algorithms exploit through interference.

In [ ]:
plus = apply_gate(ket0, H)   # |+>
minus = apply_gate(plus, Z)  # |-> = Z|+>

print("|+> amplitudes:", plus)
print("|-> amplitudes:", minus)
print()

probs_plus = amplitudes_to_probabilities(plus)
probs_minus = amplitudes_to_probabilities(minus)
print("|+> probabilities:", probs_plus)
print("|-> probabilities:", probs_minus)
print("\nProbabilities identical?", np.allclose(probs_plus, probs_minus))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

# |+> amplitudes
plot_real_amplitudes(plus, labels=['|0>', '|1>'], title='|+> amplitudes', ax=axes[0])

# |-> amplitudes
plot_real_amplitudes(minus, labels=['|0>', '|1>'], title='|-> amplitudes', ax=axes[1])

fig.suptitle('Same probabilities, different amplitudes', fontsize=12, y=1.05)
fig.tight_layout()
fig

The green and red bars show the sign difference.  If we only looked at
probabilities (squaring the amplitudes), the two states would be
indistinguishable.  But the sign carries information that *does* affect
subsequent interference.

## 6. The CNOT gate: entangling two qubits

CNOT (controlled-NOT) is a 2-qubit gate. It flips the second qubit (target)
if and only if the first qubit (control) is $|1\rangle$:

$$\text{CNOT}|00\rangle = |00\rangle, \quad \text{CNOT}|01\rangle = |01\rangle$$
$$\text{CNOT}|10\rangle = |11\rangle, \quad \text{CNOT}|11\rangle = |10\rangle$$

In [ ]:
print("CNOT matrix:")
print(CNOT)
print(f"\nIs unitary? {is_unitary(CNOT)}")
print(f"Shape: {CNOT.shape}")

In [ ]:
labels_2q = pretty_basis_labels(2)

# CNOT on each basis state
for bits in ['00', '01', '10', '11']:
    state_in = basis_state_bits(bits)
    state_out = apply_gate(state_in, CNOT)
    # Find which basis state the output is
    out_idx = int(np.argmax(np.abs(state_out)))
    print(f"CNOT|{bits}> = |{labels_2q[out_idx]}>")

### Creating entanglement

A powerful use of CNOT is creating **entangled** states -- states that cannot
be written as a tensor product of individual qubit states.

Apply H to the first qubit, then CNOT:

$$|00\rangle \xrightarrow{H \otimes I} \frac{1}{\sqrt{2}}(|00\rangle + |10\rangle) \xrightarrow{\text{CNOT}} \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

This is a **Bell state** -- maximally entangled.

In [ ]:
# Start with |00>
state = basis_state_bits('00')

# Apply H to qubit 0 (expand to 2-qubit space)
H_on_q0 = expand_single_qubit_gate(H, target=0, num_qubits=2)
state = apply_gate(state, H_on_q0)
print("After H on qubit 0:", state)

# Apply CNOT
state = apply_gate(state, CNOT)
print("After CNOT:        ", state)

# Show probabilities
probs = amplitudes_to_probabilities(state)
for lbl, p in zip(labels_2q, probs):
    print(f"  P(|{lbl}>) = {p:.4f}")

In [ ]:
fig = plot_probabilities(probs, labels=labels_2q,
                         title="Bell State: (|00> + |11>) / sqrt(2)")
fig

The only non-zero probabilities are for $|00\rangle$ and $|11\rangle$.  If you
measure one qubit, you instantly know the other -- they are perfectly
correlated.  This is entanglement in action.

## 7. expand_single_qubit_gate

When working with multi-qubit states, a single-qubit gate needs to be
"expanded" to act on the full Hilbert space.  `expand_single_qubit_gate`
builds the full $2^n \times 2^n$ matrix by tensoring with identity matrices
on the other qubits.

In [ ]:
# Z on qubit 1 in a 2-qubit system: I (x) Z
Z_on_q1 = expand_single_qubit_gate(Z, target=1, num_qubits=2)
print("Z on qubit 1 (full 4x4 matrix):")
print(Z_on_q1)
print(f"\nIs unitary? {is_unitary(Z_on_q1)}")

In [ ]:
# Apply Z to qubit 1 of the Bell state
bell = (basis_state_bits('00') + basis_state_bits('11')) / np.sqrt(2)
after_Z = apply_gate(bell, Z_on_q1)

print("Bell state:  ", bell)
print("After Z on q1:", after_Z)
print()
print("Probabilities unchanged?",
      np.allclose(amplitudes_to_probabilities(bell),
                  amplitudes_to_probabilities(after_Z)))

Again, the Z gate only changes a sign (phase), not the probabilities.  But
this phase difference *will* matter when subsequent operations cause
interference.

## 8. Demo C: The Oracle Phase Flip

This is the key insight behind Grover's algorithm and many other quantum
algorithms.

**Setup:** Start with an equal superposition over $N$ states. Mark one state
as the "target" by flipping its amplitude sign (the **oracle**).

**The puzzle:** The oracle changes only a *sign*.  The probabilities are
completely unchanged -- every state still has probability $1/N$.  If we
measured now, we would have no advantage.

**The trick:** Apply the **diffusion operator** (which reflects amplitudes
about their mean).  Because the target amplitude is now *negative* while
all others are positive, the reflection *boosts* the target amplitude and
reduces the rest.  The phase flip becomes a probability increase.

Let's watch this happen step by step.

In [ ]:
# Setup: 8 states (3 qubits), target is state 3
N = 8
target = 3
labels = pretty_basis_labels(3)

# Step 1: Equal superposition
before = equal_superposition(N)
print("Before oracle:")
print("  Amplitudes:", np.real(before).round(4))
print("  Probabilities:", amplitudes_to_probabilities(before).round(4))

# Step 2: Apply oracle (flip sign of target)
oracle = phase_oracle(target, N)
after_oracle = apply_gate(before, oracle)
print("\nAfter oracle:")
print("  Amplitudes:", np.real(after_oracle).round(4))
print("  Probabilities:", amplitudes_to_probabilities(after_oracle).round(4))

# Step 3: Apply diffusion
diffusion = diffusion_operator(N)
after_diffusion = apply_gate(after_oracle, diffusion)
print("\nAfter diffusion:")
print("  Amplitudes:", np.real(after_diffusion).round(4))
print("  Probabilities:", amplitudes_to_probabilities(after_diffusion).round(4))

In [ ]:
# Compare probabilities before and after oracle
probs_before = amplitudes_to_probabilities(before)
probs_after_oracle = amplitudes_to_probabilities(after_oracle)
probs_after_diffusion = amplitudes_to_probabilities(after_diffusion)

print("Probabilities unchanged after oracle?",
      np.allclose(probs_before, probs_after_oracle))
print(f"Target probability: before={probs_before[target]:.4f}, "
      f"after oracle={probs_after_oracle[target]:.4f}, "
      f"after diffusion={probs_after_diffusion[target]:.4f}")
print(f"\nTarget probability INCREASED from {probs_before[target]:.4f} to "
      f"{probs_after_diffusion[target]:.4f} after just one iteration!")

In [ ]:
# The full Demo C visualization
fig = plot_oracle_phase_demo(
    before_oracle=before,
    after_oracle=after_oracle,
    after_diffusion=after_diffusion,
    target_index=target,
    labels=labels,
    title="Demo C: Oracle Phase Flip Is Invisible -- Until Diffusion"
)
fig

### Reading the Demo C plot

- **Top row (amplitudes):** The oracle flips the sign of the target (orange
  bar), but diffusion then amplifies the target and suppresses the rest.
- **Bottom row (probabilities):** Before and after the oracle, all bars are
  identical at $1/N$.  After diffusion, the target bar is noticeably taller.

The oracle "tagged" the target with a phase; the diffusion operator
converted that phase tag into a probability boost.  Repeating this
oracle-diffusion cycle is Grover's algorithm, which we explore in the
next notebook.

## Key takeaways

- Quantum gates are **unitary matrices**: they preserve norms and are reversible.
- **H** creates superposition, **X** flips bits, **Z** flips the phase of $|1\rangle$.
- Phase changes are **invisible to measurement** but **crucial for interference**.
- **CNOT** entangles qubits: the resulting state cannot be factored into individual qubit states.
- `expand_single_qubit_gate` embeds a 1-qubit gate into a multi-qubit Hilbert space.
- **Demo C:** An oracle's phase flip does not change probabilities, but the diffusion operator converts the hidden phase information into an observable probability increase.

**Next notebook:** Grover's algorithm -- repeated oracle + diffusion iterations as geometric rotation.